# 06 - Submission Charts

The headline tables for the submission, assembled from the live
event log and run report: identity, regime time, drawdown summary,
and per-asset attribution. Everything reuses `guardrail_lab` and
renders as text/Markdown so it runs with the base stack.

**Offline-safe:** every section degrades to a hint when data is
absent.


In [ ]:
# --- Guardrail Alpha notebook bootstrap (offline-safe) ---
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Return the first ancestor that contains python-lab/guardrail_lab."""
    for candidate in [start, *start.parents]:
        if (candidate / "python-lab" / "guardrail_lab").is_dir():
            return candidate
    return start


# In a notebook __file__ is undefined, so anchor on the current working dir.
REPO_ROOT = _find_repo_root(Path.cwd())
LAB_PATH = REPO_ROOT / "python-lab"
if str(LAB_PATH) not in sys.path:
    sys.path.insert(0, str(LAB_PATH))


def _first_existing(*candidates: Path):
    """Return the first path that exists, else None."""
    for path in candidates:
        if path.exists():
            return path
    return None


DATA_DIR = REPO_ROOT / "data"
CONFIG_DIR = REPO_ROOT / "configs"

# Prefer a real run; fall back to the seeded demo artifacts if present.
DB_PATH = _first_existing(
    DATA_DIR / "guardrail_alpha.db",
    DATA_DIR / "demo_guardrail_alpha.db",
)
RUN_REPORT_PATH = _first_existing(
    DATA_DIR / "run_report.json",
    DATA_DIR / "demo_run_report.json",
)
EXPERIMENTS_DIR = DATA_DIR / "experiments"
BACKTESTS_DIR = DATA_DIR / "backtests"
ELIGIBLE_ASSETS_PATH = CONFIG_DIR / "eligible_assets.bsc.json"
ASSET_CATEGORIES_PATH = CONFIG_DIR / "asset_categories.json"

NO_DATA_HINT = (
    "No data found under data/. Run the agent (paper/live) or seed a demo run "
    "first, e.g.  python3 -m guardrail_lab.seed  (from python-lab/), then "
    "re-run this notebook. The notebook is offline-safe and will not raise."
)

print("repo root :", REPO_ROOT)
print("event log :", DB_PATH if DB_PATH else "(none yet)")
print("run report:", RUN_REPORT_PATH if RUN_REPORT_PATH else "(none yet)")


In [ ]:
def render_table(rows, columns, title=None):
    """Print an aligned text table. rows: list[dict]; columns: list[(key,label)]."""
    if title:
        print(title)
        print("=" * len(title))
    if not rows:
        print("(no rows)")
        return
    labels = [label for _, label in columns]
    keys = [key for key, _ in columns]
    cells = [[("" if r.get(k) is None else str(r.get(k))) for k in keys] for r in rows]
    widths = [
        max(len(labels[i]), *(len(row[i]) for row in cells)) for i in range(len(keys))
    ]
    header = "  ".join(labels[i].ljust(widths[i]) for i in range(len(keys)))
    print(header)
    print("  ".join("-" * widths[i] for i in range(len(keys))))
    for row in cells:
        print("  ".join(row[i].ljust(widths[i]) for i in range(len(keys))))


## Load everything


In [ ]:
from guardrail_lab.db import load_events, event_counts
from guardrail_lab.loaders import load_run_report
from guardrail_lab.metrics import nav_series, max_drawdown, trade_count
from guardrail_lab.regime_analysis import time_in_regime
from guardrail_lab.drawdown import analyze_drawdown_from_events
from guardrail_lab.attribution import trade_attribution

events = load_events(str(DB_PATH)) if DB_PATH else []
report = load_run_report(str(RUN_REPORT_PATH)) if RUN_REPORT_PATH else None

if not events and report is None:
    print(NO_DATA_HINT)
else:
    print(f"Events: {len(events)}  |  Run report: "
          f"{'loaded' if report else '(none)'}")


## Headline: agent identity & status


In [ ]:
if report:
    ident = [
        {"field": "run_id", "value": report.get("run_id", "n/a")},
        {"field": "mode", "value": report.get("mode", "n/a")},
        {"field": "wallet_address", "value": report.get("wallet_address", "n/a")},
        {"field": "policy_hash", "value": report.get("policy_hash", "n/a")},
        {"field": "kill_switch", "value": report.get("kill_switch", "n/a")},
        {"field": "starting_nav_usd", "value": report.get("starting_nav_usd", "n/a")},
        {"field": "nav_usd", "value": report.get("nav_usd", "n/a")},
        {"field": "total_drawdown_pct", "value": report.get("total_drawdown_pct", "n/a")},
    ]
    render_table(ident, [("field", "FIELD"), ("value", "VALUE")],
                 title="Agent identity & status")
else:
    print("No run report available;", NO_DATA_HINT)


## Headline: performance summary


In [ ]:
if events:
    curve = nav_series(events)
    navs = [nav for _, nav in curve]
    perf = []
    if navs:
        ret = (navs[-1] - navs[0]) / navs[0] * 100.0 if navs[0] else 0.0
        perf.append({"metric": "starting_nav_usd", "value": f"{navs[0]:,.2f}"})
        perf.append({"metric": "latest_nav_usd", "value": f"{navs[-1]:,.2f}"})
        perf.append({"metric": "total_return_pct", "value": f"{ret:+.3f}%"})
        perf.append({"metric": "max_drawdown_pct", "value": f"{max_drawdown(navs) * 100.0:.3f}%"})
    perf.append({"metric": "confirmed_trades", "value": str(trade_count(events))})
    render_table(perf, [("metric", "METRIC"), ("value", "VALUE")],
                 title="Performance summary")
else:
    print(NO_DATA_HINT)


## Headline: time in regime


In [ ]:
if events:
    rows = time_in_regime(events)
    if rows:
        render_table(
            [
                {
                    "regime": t.regime,
                    "classifications": t.classifications,
                    "share": f"{t.fraction:.1%}",
                    "seconds": f"{t.seconds:.1f}",
                }
                for t in rows
            ],
            [
                ("regime", "REGIME"),
                ("classifications", "COUNT"),
                ("share", "SHARE"),
                ("seconds", "SECONDS"),
            ],
            title="Time in regime",
        )
    else:
        print("No regime classifications in the log yet.")
else:
    print(NO_DATA_HINT)


## Headline: drawdown summary


In [ ]:
if events:
    dd = analyze_drawdown_from_events(events, top_n=3)
    if dd.points:
        print(f"Max drawdown: {dd.max_drawdown_pct:.4f}%  "
              f"(peak {dd.peak_timestamp or 'n/a'} -> trough {dd.trough_timestamp or 'n/a'})")
        if dd.episodes:
            render_table(
                [
                    {
                        "depth": f"{ep.depth_pct:.4f}%",
                        "trough": ep.trough_timestamp,
                        "recovered": ep.recovered,
                    }
                    for ep in dd.episodes
                ],
                [("depth", "DEPTH"), ("trough", "TROUGH_TS"), ("recovered", "RECOVERED")],
                title="Top drawdown episodes",
            )
        else:
            print("No drawdown episodes (no decline below a prior peak).")
    else:
        print("No NAV history for drawdown analysis yet.")
else:
    print(NO_DATA_HINT)


## Headline: trade attribution summary


In [ ]:
if events:
    attribution = trade_attribution(events)
    if attribution:
        total_usd = sum(a["total_amount_usd"] for a in attribution)
        render_table(
            [
                {
                    "symbol": a["symbol"],
                    "trades": a["count"],
                    "notional_usd": f"{a['total_amount_usd']:,.2f}",
                    "share": f"{(a['total_amount_usd'] / total_usd * 100.0):.1f}%" if total_usd else "n/a",
                }
                for a in attribution
            ],
            [
                ("symbol", "SYMBOL"),
                ("trades", "TRADES"),
                ("notional_usd", "NOTIONAL_USD"),
                ("share", "SHARE"),
            ],
            title="Trade attribution",
        )
    else:
        print("No confirmed trades in the log yet.")
else:
    print(NO_DATA_HINT)


## Notes

These tables are the offline-safe, dependency-free version of the
submission headline charts. For richer HTML/PNG output the
`guardrail_lab.submission` and `guardrail_lab.charts` modules build
on the same underlying functions when the optional plotting stack
is installed.
